# Text2Event-mT5 — BKEE
Default: **smoke, then full training only if smoke passes** (authorized by user). Enable GPU and Internet. Attach the private input archive
`text2event-kaggle.zip` built with `scripts/package_text2event.py`.
This archive includes runtime scripts, so the notebook does not depend on unpushed Git files.

Variant: mT5, relaxed event–role constraints, trigger-first curriculum, heuristic spans.
Checkpoint resume is at epoch boundaries. Save notebook outputs before the session expires;
attach the previous output and set RESUME_FROM to its training output directory.
No TEST evaluation for checkpoint selection. Smoke scores are not baseline results.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.14.1', 'sentencepiece==0.2.1'], check=True)
# Keep Kaggle's CUDA-enabled torch installation; record it in smoke-report.json.


In [ ]:
DATASET = 'BKEE'
MODE = 'smoke_then_train'
APPROVED_FULL_RUN = True
EPOCHS = 30  # provisional; review after GPU smoke
TRIGGER_EPOCHS = 1
PATIENCE = 5
MIN_DELTA = 0.0  # DEV AC on scale 0..1
MICRO_BATCH = 1
ACCUMULATION = 8
SEED = 42
RESUME_FROM = None  # trusted previous output directory, only for MODE='train'


In [ ]:
from pathlib import Path
import json, hashlib, zipfile, os, shutil
archives = list(Path('/kaggle/input').rglob('text2event-kaggle.zip'))
extracted = list(Path('/kaggle/input').rglob('package-manifest.json'))
ROOT = Path('/kaggle/working/text2event')
if ROOT.exists(): raise FileExistsError('Use a fresh session')
if len(extracted) == 1:
    shutil.copytree(extracted[0].parent, ROOT)
elif len(archives) == 1:
    ROOT.mkdir()
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (ROOT/member.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(ROOT)
else:
    print('Input files:', [str(p) for p in Path('/kaggle/input').rglob('*')][:100])
    raise RuntimeError('Expected one archive or one extracted package manifest')
manifest = json.loads((ROOT/'package-manifest.json').read_text())
for name,digest in manifest.items():
    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest()!=digest:
        raise ValueError(f'Package checksum mismatch: {name}')
os.chdir(ROOT)
print('Verified',len(manifest),'files')

assert manifest['src/baselines/text2event/decoding.py'] == '2b641da7667c16ee9438e68a7d6183fb481cf92961cd5e4bd5cc4392cc4a5ba6', 'Stale runtime: attach the latest input version'


In [ ]:
import subprocess, sys, shutil
phases = ['smoke', 'train'] if MODE == 'smoke_then_train' else [MODE]
for phase in phases:
    output = Path('/kaggle/working') / f'text2event-{DATASET}-{phase}'
    if phase == 'smoke':
        command = [sys.executable, '-m', 'scripts.smoke_text2event', '--bundle', str(ROOT/'bundle'),
                   '--dataset', DATASET, '--output', str(output), '--stress-max-length']
    elif phase == 'train':
        if not APPROVED_FULL_RUN:
            raise RuntimeError('Review smoke results and configuration before enabling full training')
        if RESUME_FROM:
            if output.exists(): raise FileExistsError(output)
            shutil.copytree(RESUME_FROM, output)
        command = [sys.executable, '-m', 'scripts.train_text2event', '--bundle', str(ROOT/'bundle'),
                   '--dataset', DATASET, '--output', str(output), '--epochs', str(EPOCHS),
                   '--trigger-epochs', str(TRIGGER_EPOCHS), '--patience', str(PATIENCE), '--min-delta', str(MIN_DELTA), '--micro-batch', str(MICRO_BATCH),
                   '--accumulation', str(ACCUMULATION), '--seed', str(SEED),
                   '--gradient-checkpointing', '--final-evaluation']
        if RESUME_FROM: command += ['--resume']
    else:
        raise ValueError('MODE must be smoke or train')
    print(' '.join(command))
    with (Path('/kaggle/working') / f'{DATASET}-{phase}.log').open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        if process.wait() != 0:
            raise RuntimeError('Run failed; inspect saved log and smoke-report.json. Do not silently truncate data.')
    if phase == 'smoke':
        report = json.loads((output/'smoke-report.json').read_text())
        print(json.dumps(report, indent=2, ensure_ascii=False))
        if report['status'] != 'PASS':
            raise RuntimeError('Smoke is not PASS; review diagnostics before training')
        # This derived checkpoint is no longer needed; keep reports and predictions.
        (output/'last.pt').unlink(missing_ok=True)
